> 在线讲义：[第 30 章 · 推理模式与 KV cache：生成一个 token 要做什么](https://wikieden.github.io/pytorch-inside-out/lessons/30-inference-mode.html) · 本书《PyTorch 从里到外》

# 第 30 章 · 推理模式与 KV cache：生成一个 token 要做什么

本章回答：推理时 `eval()`、`no_grad()`、`inference_mode()` 各关掉了什么？语言模型一个一个 token 往外吐，为什么要 KV cache，它有多大？为什么 decode 阶段是访存受限，batch 1 的速度上限怎么算？静态 cache + CUDA Graph 是怎么回事？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/30-inference-mode.md`（讲解、动画、练习都在讲义里）。对应原文：gpt-fast 仓库与博客「Accelerating Generative AI with PyTorch II: GPT, Fast」（〔在线〕pytorch.org/blog/accelerating-generative-ai-2/）；LLM 全链路地图（`llm-full-stack/notes/00-full-stack-map.md`）推理部分；Transformer 讲义 第 17 章 因果 mask；本章正确性和 inference_mode 行为在本机 torch 2.15.0 上验证过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 inference 张量 | 不能存给反向 | `inference_tensor_error()` | Inference tensors cannot be saved for backward |
| 第二步 KV cache | 一步步算 = 一次算 | `kv_cache_matches()` | True |
| 第三步 cache 大小 | 2·L·n_kv·d·2 | `kv_bytes_per_token(32, 32, 128)` | 524288（0.5 MB），GQA 131072 |
| 第四步 decode 上限 | 带宽 ÷ 权重字节 | `decode_ceiling(3.35e12, 14e9)` | 239 tok/s |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `decode_ceiling`**（来自 `ch30_inference_mode.py`）：下面要用到的小工具。

In [ ]:
def decode_ceiling(bandwidth: float, weight_bytes: float) -> float:
    return bandwidth / weight_bytes

**函数 `inference_tensor_error`**（来自 `ch30_inference_mode.py`）：下面要用到的小工具。

In [ ]:
def inference_tensor_error() -> str:
    m = nn.Linear(10, 10)
    with torch.inference_mode():
        b = m(torch.randn(2, 10))
    try:
        m(b).sum().backward()
        return "没报错"
    except RuntimeError as e:
        return str(e).split(".")[0]

**函数 `kv_bytes_per_token`**（来自 `ch30_inference_mode.py`）：下面要用到的小工具。

In [ ]:
def kv_bytes_per_token(layers: int, kv_heads: int, head_dim: int, bytes_per: int = 2) -> int:
    return 2 * layers * kv_heads * head_dim * bytes_per

**函数 `kv_cache_matches`**（来自 `ch30_inference_mode.py`）：一步步用 KV cache 算的输出 == 一次性因果 attention。

In [ ]:
def kv_cache_matches(T: int = 6, d: int = 8, seed: int = 0) -> bool:
    """一步步用 KV cache 算的输出 == 一次性因果 attention。"""
    torch.manual_seed(seed)
    x = torch.randn(T, d)
    Wq, Wk, Wv = (torch.randn(d, d) for _ in range(3))
    full = F.scaled_dot_product_attention((x @ Wq)[None], (x @ Wk)[None], (x @ Wv)[None], is_causal=True)[0]
    K, V, outs = [], [], []
    for t in range(T):
        q = x[t : t + 1] @ Wq
        K.append(x[t : t + 1] @ Wk)
        V.append(x[t : t + 1] @ Wv)
        outs.append(F.scaled_dot_product_attention(q[None], torch.cat(K)[None], torch.cat(V)[None])[0])
    return torch.allclose(torch.cat(outs), full, atol=1e-5)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 inference 张量拿去训练：", inference_tensor_error())

In [ ]:
print("\n第二步 KV cache 与整段计算一致：", kv_cache_matches())
llama2 = kv_bytes_per_token(32, 32, 128)

In [ ]:
print("\n第三步 Llama-2-7B 每 token", llama2, "字节，4096 token", llama2 * 4096 / 2**30, "GiB；GQA 8 个 KV 头",
      kv_bytes_per_token(32, 8, 128), "字节")

In [ ]:
print("\n第四步 7B bf16 在 H100 上 decode 上限 ≈", round(decode_ceiling(3.35e12, 14e9)), "tok/s")

## 3. 代码里才看得到的

1. **带 cache 的那一步不用 `is_causal`**：新 token 本来就只能看到前面已经存进 cache 的。
2. **`torch.cat` 每步新分配显存**：这正是静态 cache 要改掉的（第五步）。

## 4. 自己改一改

- 把 `kv_cache_matches` 的 T 改成 20。
- 算 Llama-3-70B（80 层、8 个 KV 头、128 维）每 token 的 cache。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 推理开 eval 加 inference_mode：切层的行为、关记账和版本号；
> 因果 attention 旧 KV 不变，存进 cache 每步只算新 token；
> 每 token cache 是 2 乘层乘 KV 头乘头维乘字节，GQA 共享省几倍；
> prefill 吃算力、decode 吃带宽，上限是带宽除以权重字节；静态 cache 才能录 CUDA Graph。